<a href="https://colab.research.google.com/github/ajndantas/Conselheiro-de-Saude-Mental/blob/master/Conselheiro_de_Saude_Mental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install google-genai
%pip -q install google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.

In [ ]:
from os import environ
from google.colab import userdata

api_key = input('Por favor, digite o nome de sua chave de API: ')
environ[api_key] = userdata.get(api_key)

Por favor, digite o nome de sua chave de APIGOOGLE_API_KEY


In [ ]:
from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
#from google.adk.tools.image_generation import ImageGenerationTool
from google.genai import types
from datetime import date
import textwrap
from IPython.display import display, Markdown
import requests
import warnings

warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'google.adk.tools.image_generation'

In [ ]:
def call_agent(agent: Agent, message_text: str) -> str:
    session_service = InMemorySessionService()
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""

    for event in runner.run(user_id="user1", session_id="session1", new_message=content):

        if event.is_final_response():
            for part in event.content.parts:
                if part.text is not None:

                    final_response += part.text

                    final_response += "\n"

    return final_response

In [ ]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
##########################################
# ---     Agente 1: Psicólogo        --- #
##########################################

def agente_psicologo(resposta):
    psicologo = Agent(
        # Inserir as instruções do Agente Planejador #################################################
        name="psicologo",
        model="gemini-2.0-flash",
        instruction="""
        Aja como um psicólogo e seja empático. Você deve fazer o seguinte:
        1 - Dar conselhos para alguém que está com a saúde mental debilitada conforme a resposta dada.
        2 - Se a resposta informada for um sentimento positivo, responda para que a pessoa aproveite a vida e seja feliz.
        3 - Informar 5 possíveis transtornos mentais causadores da resposta dada usando o Google (google_search). Se a transtorno mental tiver poucas reações entusiasmadas, é possível que ele não seja tão relevante assim e
        possa ser substituída por outra que tenha mais.
        """,
        description="Agente psicólogo",
        tools=[google_search] # Utilizando a ferramenta de busca do Google importada anteriormente do ADK
    )

    entrada_do_agente_psicologo = f"Resposta:{resposta}\n"
    conselhos = call_agent(psicologo, entrada_do_agente_psicologo)

    return conselhos

In [ ]:
################################################
# --- Agente 2: Atividades --- #
################################################
# SOBRE AS NOTICIAS, DECIDA QUAL NOTÍCIA É A QUE TEM MAIOR RELEVÂNCIA E SEUS PONTOS POSITVOS E NEGATVOS

def agente_atividades(conselhos):
    atividades = Agent(
        name="atividades",
        model="gemini-2.0-flash",
        instruction="""
        Você é um psicologo, e reaja de maneira empática. Você deve:
        1 - Ser capaz de sugerir no máximo 5 atividades de relaxamento ou mindfullness.
        Se a atividade tiver poucas reações entusiasmadas, é possível que ele não seja tão relevante assim e
        possa ser substituída por outra que tenha mais.
        """,
        description="Lista atividades de relaxamento e mindfullness",
        tools=[google_search]
    )

    entrada_do_agente_atividades = f"Conselhos:{conselhos}\n"
    atividades = call_agent(atividades, entrada_do_agente_atividades)

    return atividades



In [ ]:
print("🚀 Iniciando o Conselheiro de Saúde Mental 🚀")

resposta = input("❓ Como você está se sentindo hoje ? ")

# Inserir lógica do sistema de agentes ################################################
if not resposta:
    print("Nenhuma resposta foi fornecida")
else:
    print(f"🔍 Vamos então obter os conselhos de nosso psicólogo virtual em função do que você respondeu {resposta}")

    conselhos = agente_psicologo(resposta)
    print("\n 📝 Conselhos\n")
    display(to_markdown(conselhos))
    print("-------------------------------------------------")

    atividades = agente_atividades(conselhos)
    print("\n 📝 Atividades\n")
    display(to_markdown(atividades))
    print("-------------------------------------------------")

🚀 Iniciando o Conselheiro de Saúde Mental 🚀
❓ Como você está se sentindo hoje ? triste
🔍 Vamos então obter os conselhos de nosso psicólogo virtual em função do que você respondeu triste

 📝 Conselhos



> Sinto muito que você esteja se sentindo triste. É importante reconhecer e validar seus sentimentos. A tristeza pode ser um sinal de que algo precisa de atenção em sua vida.
> 
> **Aqui estão alguns conselhos que podem ajudar:**
> 
> 1.  **Permita-se sentir:** Não reprima a tristeza. Chorar, escrever em um diário ou conversar com alguém de confiança pode ajudar a liberar suas emoções.
> 2.  **Cuide de si mesmo:** Priorize o autocuidado. Isso inclui dormir o suficiente, alimentar-se de forma saudável e praticar exercícios físicos. Pequenas mudanças na rotina podem ter um grande impacto no seu bem-estar.
> 3.  **Conecte-se com os outros:** O isolamento pode intensificar a tristeza. Procure estar perto de pessoas que te apoiam e te fazem sentir bem. Compartilhe seus sentimentos com amigos, familiares ou um terapeuta.
> 4.  **Identifique a causa:** Tente entender o que está causando a sua tristeza. Pode ser um evento específico, um padrão de pensamento negativo ou uma combinação de fatores.
> 5.  **Procure ajuda profissional:** Se a tristeza persistir por mais de duas semanas ou estiver interferindo em sua vida diária, considere procurar um psicólogo ou psiquiatra. Eles podem te ajudar a entender seus sentimentos e desenvolver estratégias para lidar com eles.
> 
> Lembre-se que você não está sozinho e que há esperança. A tristeza é uma emoção humana normal, mas não precisa te dominar.
> 
> Para te ajudar a entender melhor o que você pode estar sentindo, vou pesquisar alguns possíveis transtornos mentais que podem estar associados à tristeza:
> 
> 
> Com base nas informações que encontrei, aqui estão 5 possíveis transtornos mentais que podem estar associados à tristeza:
> 
> 1.  **Depressão:** É um transtorno comum que causa tristeza persistente, perda de interesse e dificuldade em realizar tarefas diárias.
> 2.  **Transtorno Bipolar:** Doença psiquiátrica que provoca oscilações imprevisíveis no humor, variando entre depressão e mania.
> 3.  **Transtorno de Ansiedade:** Pode levar ao esgotamento emocional, resultando em tristeza e vontade de chorar.
> 4.  **Transtorno de Estresse Pós-Traumático (TEPT):** A ansiedade decorrente de um evento traumático pode causar sofrimento psicológico e tristeza.
> 5.  **Transtorno Depressivo Persistente (Distimia):** Caracterizado por humor deprimido na maior parte do dia, por um longo período de tempo.
> 
> Lembre-se que esta é apenas uma lista de possíveis causas e que um diagnóstico preciso só pode ser feito por um profissional de saúde mental. Se você está preocupado com a sua tristeza, procure ajuda.
> 


-------------------------------------------------

 📝 Atividades



> Sinto muito que você esteja passando por um momento difícil. É muito bom que você esteja buscando maneiras de lidar com a tristeza e cuidar de si mesmo. Reconhecer e validar seus sentimentos é um passo importante para se sentir melhor.
> 
> Além dos conselhos que você já recebeu, gostaria de sugerir algumas atividades de relaxamento e mindfulness que podem ajudar a aliviar a tristeza e promover o bem-estar:
> 
> 1.  **Meditação Mindfulness:** Reserve alguns minutos do seu dia para se concentrar na sua respiração e observar seus pensamentos e sentimentos sem julgamento. Existem diversos aplicativos e vídeos online que podem te guiar nesse processo. A meditação mindfulness pode ajudar a reduzir o estresse, a ansiedade e a melhorar o humor.
> 2.  **Yoga:** A prática de yoga combina posturas físicas, técnicas de respiração e meditação, o que pode ajudar a relaxar o corpo e a mente. Existem diversos estilos de yoga, então você pode experimentar diferentes opções para encontrar aquela que mais te agrada. A yoga pode ajudar a aliviar a tensão muscular, melhorar a flexibilidade e promover uma sensação de calma e bem-estar.
> 3.  **Caminhada na Natureza:** Passar tempo ao ar livre, em contato com a natureza, pode ter um efeito positivo no seu humor e bem-estar. Caminhar em um parque, floresta ou praia pode ajudar a reduzir o estresse, a ansiedade e a melhorar a concentração. Observe as cores, os sons e os cheiros ao seu redor, e permita-se relaxar e aproveitar o momento presente.
> 4.  **Escrita Terapêutica:** Escrever sobre seus pensamentos e sentimentos pode ser uma forma de processar suas emoções e encontrar clareza. Você pode escrever em um diário, criar poemas ou contos, ou simplesmente anotar o que vem à sua mente. A escrita terapêutica pode ajudar a liberar emoções reprimidas, a identificar padrões de pensamento negativos e a promover o autoconhecimento.
> 5.  **Ouvir Música Relaxante:** A música pode ter um efeito poderoso no seu humor e bem-estar. Ouvir músicas calmas e relaxantes pode ajudar a reduzir o estresse, a ansiedade e a promover uma sensação de paz e tranquilidade. Experimente diferentes gêneros musicais e encontre aqueles que mais te agradam e te fazem sentir bem.
> 
> Lembre-se que cada pessoa é única e o que funciona para uma pessoa pode não funcionar para outra. Experimente diferentes atividades e descubra aquelas que te trazem mais conforto e bem-estar. Seja gentil consigo mesmo e permita-se tempo para se curar e se recuperar.
> 
> É importante ressaltar que essas atividades não substituem o acompanhamento profissional. Se a tristeza persistir ou estiver interferindo em sua vida diária, procure um psicólogo ou psiquiatra. Eles podem te ajudar a entender seus sentimentos e desenvolver estratégias para lidar com eles.
> 


-------------------------------------------------
